# Token Budgets
> Allocate and manage token limits across different context sources.

`TokenBudget` lets you define a total token ceiling and carve it into
named allocations (system prompt, memory, retrieval, etc.) with individual
limits, priorities, and overflow strategies.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor import TokenBudget, BudgetAllocation, SourceType

## Create a Token Budget
Define a budget with three allocations and a reserve for the model’s output.

In [ ]:
budget = TokenBudget(
    total_tokens=8192,
    allocations=[
        BudgetAllocation(
            source=SourceType.SYSTEM,
            max_tokens=500,
            priority=10,
            overflow_strategy="truncate",
        ),
        BudgetAllocation(
            source=SourceType.MEMORY,
            max_tokens=2000,
            priority=7,
            overflow_strategy="truncate",
        ),
        BudgetAllocation(
            source=SourceType.RETRIEVAL,
            max_tokens=4000,
            priority=5,
            overflow_strategy="drop",
        ),
    ],
    reserve_tokens=200,
)

print(f"Total budget: {budget.total_tokens} tokens")
print(f"Reserve:      {budget.reserve_tokens} tokens")
print(f"Allocations:  {len(budget.allocations)}")

## Query Individual Allocations
Use `.get_allocation()` to look up the token limit for a specific source.
Access individual `BudgetAllocation` objects through the `.allocations` list.

In [ ]:
# get_allocation returns the max_tokens int for a source
memory_tokens = budget.get_allocation(SourceType.MEMORY)
print(f"Memory allocation: {memory_tokens} tokens")

# Access the full BudgetAllocation object from the list
memory_alloc = next(a for a in budget.allocations if a.source == SourceType.MEMORY)
print(f"Source:   {memory_alloc.source.value}")
print(f"Max:      {memory_alloc.max_tokens} tokens")
print(f"Priority: {memory_alloc.priority}")
print(f"Overflow: {memory_alloc.overflow_strategy}")

## Overflow Strategies
Each allocation declares what happens when its content exceeds the limit:
- `"truncate"` – cut content to fit
- `"drop"` – discard the entire block

In [ ]:
strategy = budget.get_overflow_strategy(SourceType.RETRIEVAL)
print(f"Retrieval overflow strategy: {strategy}")

strategy_sys = budget.get_overflow_strategy(SourceType.SYSTEM)
print(f"System overflow strategy:    {strategy_sys}")

## Shared Pool
Tokens not assigned to any allocation form a shared pool available to
lower-priority sources.

In [ ]:
shared = budget.shared_pool

allocated = sum(a.max_tokens for a in budget.allocations)
print(f"Allocated tokens: {allocated}")
print(f"Reserve tokens:   {budget.reserve_tokens}")
print(f"Shared pool:      {shared} tokens")

## Key Takeaways
- `TokenBudget` enforces a hard ceiling on total context size
- `BudgetAllocation` assigns per-source limits with priority ordering
- `overflow_strategy` controls truncation vs. dropping behavior
- The shared pool absorbs leftover capacity for flexible use

**Next:** [Budget Presets](02_budget_presets.ipynb)